In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
    REPO_PATH = "/content/packt-modern-time-series"

    if not os.path.exists(REPO_PATH):
        os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

    os.chdir(f"{REPO_PATH}/student_notebooks")

    if REPO_PATH not in sys.path:
        sys.path.insert(0, REPO_PATH)

print(f"✓ Setup complete — {os.getcwd()}")

---
# Module 7 — Prediction Intervals
**Type:** [Watch Only / Light Execution]  
**Time:** 25 minutes  
**Job:** Judge uncertainty quality across all models. Score 80% intervals. Visualize fan charts. Build the uncertainty leaderboard.

---
## 7.1 — Setup
**[Watch Only]**

---

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from config import (
    ARTIFACT_DIR,
    WORKSHOP_SUBSET_N,
    MICRO_SUBSET_N,
    INTERVAL_COVERAGE,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts, pooled_interval_score, coverage_rate
from src.schemas import validate_score
from src.plotting import (
    plot_interval_grid,
    plot_interval_width_distribution,
    plot_metric_scatter,
)

print("Setup complete.")

---
## 7.2 — Why Intervals Matter
**[Watch Only]**

**Point forecasts are incomplete.** A model that predicts 100 units cannot tell you how much safety stock to hold.

**The procurement scenario:**  
- Model predicts 100 units; 80% interval is [90, 110] → hold 10 units of safety stock. Reasonable.  
- Same prediction, 80% interval is [10, 500] → you need 400 units to cover the upside, or accept a 20% stockout risk. That forecast is operationally useless.

**Why wMAPE doesn't surface this:**  
Two models with identical wMAPE can have wildly different interval quality. The width and calibration of the interval — not just the point — determines whether the forecast is usable for purchasing.

**What Interval Score measures:**  
- Penalizes width (even if coverage is met)  
- Penalizes misses at 10× the miss size (for 80% intervals)  
- Lower = better on both dimensions simultaneously

> **Key idea:** The uncertainty leaderboard will look different from the wMAPE leaderboard. That difference is the information you need for a production decision.

---
## 7.3 — Load All Forecast Artifacts
**[Watch Only]**

---

In [ ]:
baseline_full = load_checkpoint("04_baseline_forecasts")
ml_full       = load_checkpoint("05_ml_forecasts")
dl_full       = load_checkpoint("06_dl_forecasts")

all_forecasts = pd.concat([baseline_full, ml_full, dl_full], ignore_index=True)

models_with_intervals = [
    m for m in all_forecasts["model"].unique()
    if all_forecasts[all_forecasts["model"] == m][["lo_80", "hi_80"]].notna().all(axis=None)
]

print(f"Total forecast rows  : {len(all_forecasts):,}")
print(f"Models in panel      : {sorted(all_forecasts['model'].unique().tolist())}")
print(f"Models with intervals: {sorted(models_with_intervals)}")

**Expected output:**
```
Total forecast rows  : 504,000
Models in panel      : ['AutoETS', 'Chronos-t5-mini', 'LightGBM', 'NHITS', 'Naive', 'SeasonalNaive']
Models with intervals: ['AutoETS', 'Chronos-t5-mini', 'LightGBM', 'NHITS', 'Naive', 'SeasonalNaive']
```

---
## 7.4 — Fan Chart: One Series, All Models
**[Watch Only]**

---

In [ ]:
panel = load_checkpoint("03_validated_panel")

sample_uid = (
    panel.groupby("unique_id")["y"].sum()
    .sort_values(ascending=False)
    .index[0]
)
sample_cut = all_forecasts["cutoff"].max()

plot_interval_grid(
    actuals_df=panel,
    forecasts_df=all_forecasts,
    unique_id=sample_uid,
    models=sorted(models_with_intervals),
    cutoff=sample_cut,
    title=f"80% Prediction Intervals — {sample_uid} (Window 3)",
)
print("Wider shaded band = less certain model. Narrower band that still covers actuals = better.")

**Expected output:** A 2×3 grid of fan charts, one per model. Each shows actuals, the point forecast, and the 80% interval as a shaded band.

---
## 7.5 — Interval Width Distribution
**[Watch Only]**

---

In [ ]:
fig, ax, medians = plot_interval_width_distribution(
    forecasts_df=all_forecasts,
    models=sorted(models_with_intervals),
    title="Interval Width Distribution by Model (Full Panel, All Windows)",
)

print("Median interval widths:")
for m, w in medians.items():
    print(f"  {m:<20}: {w:.1f} units")

**Expected output:** Overlapping histograms. Tighter distributions at lower widths = more confident intervals.

---
## 7.6 — Score All Intervals
**[Watch Only]**

---

In [ ]:
all_scores = score_forecasts(
    all_forecasts,
    subset_name=f"workshop_{WORKSHOP_SUBSET_N}",
)
all_scores = validate_score(all_scores, artifact_name="07_uncertainty_leaderboard")

uncertainty_lb = (
    all_scores
    .pivot_table(index="model", columns="metric", values="score")
    .reset_index()
)
uncertainty_lb.columns.name = None

display_cols   = ["model", "IntervalScore_80", "Coverage_80", "wMAPE"]
available_cols = [c for c in display_cols if c in uncertainty_lb.columns]
uncertainty_lb[available_cols].dropna(subset=["IntervalScore_80"]).sort_values("IntervalScore_80")

**Expected output:** Leaderboard sorted by IntervalScore_80 — may differ from the wMAPE ranking.

---
## 7.7 — Coverage Diagnostic
**[Watch Only]**

---

In [ ]:
if "Coverage_80" in uncertainty_lb.columns:
    cov_data = uncertainty_lb[["model", "Coverage_80"]].dropna().sort_values("Coverage_80", ascending=False)
    fig, ax = plt.subplots(figsize=(9, 4))
    bar_colors = [
        "#43A047" if abs(v - INTERVAL_COVERAGE) < 0.05
        else ("#FF9800" if v > INTERVAL_COVERAGE else "#E53935")
        for v in cov_data["Coverage_80"]
    ]
    bars = ax.barh(cov_data["model"], cov_data["Coverage_80"],
                   color=bar_colors, edgecolor="white")
    ax.axvline(INTERVAL_COVERAGE, color="#333", linestyle="--", linewidth=1.2,
               label=f"Target: {int(INTERVAL_COVERAGE*100)}%")
    ax.set_xlabel("Coverage (fraction of actuals inside 80% interval)")
    ax.set_title("Coverage Diagnostic — 80% Prediction Intervals", fontsize=10)
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    for bar, val in zip(bars, cov_data["Coverage_80"]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.1%}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()
    print("Green = within 5% of target | Orange = conservative | Red = overconfident")
    print("Coverage is diagnostic only — not used for ranking.")

**Expected output:** Horizontal bar chart showing coverage per model. Green = well-calibrated, orange = too wide, red = too narrow.

---
## 7.8 — wMAPE vs Interval Score Scatter
**[Watch Only]**

---

In [ ]:
if "IntervalScore_80" in uncertainty_lb.columns and "wMAPE" in uncertainty_lb.columns:
    plot_metric_scatter(
        df=uncertainty_lb.dropna(subset=["wMAPE", "IntervalScore_80"]),
        x="wMAPE",
        y="IntervalScore_80",
        title="Point Accuracy vs Uncertainty Quality",
        xlabel="wMAPE (lower = better point accuracy)",
        ylabel="Interval Score 80% (lower = better uncertainty)",
    )
    print("Bottom-left = best on both dimensions.")
    print("A model can win wMAPE but lose on interval quality — and vice versa.")

**Expected output:** Scatter plot with one point per model. Models toward the bottom-left dominate on both accuracy and uncertainty.

---
## 7.9 — Save the Uncertainty Leaderboard Artifact
**[Watch Only]**

---


In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
uncertainty_path = ARTIFACT_DIR / "07_uncertainty_leaderboard.parquet"
all_scores.to_parquet(uncertainty_path, index=False)
print(f"  ✓ Uncertainty leaderboard saved: {uncertainty_path.name} ({len(all_scores):,} rows)")

**Expected output:**
```
  ✓ Uncertainty leaderboard saved: 07_uncertainty_leaderboard.parquet (18 rows)
```